# 🎲 5장 · 우연인가 아닌가: 통계적 추론 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 5장의 **05. 파이썬 실습**, **04절 계산 확인**, **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 5-1, 5-3, 5-4, 5-6, 5-7, 5-8 |
| ☆ 선택 | 자습이나 심화 | 5-2, 5-5, 5-9, 5-10, 5-11 |

**데이터**
- 따릉이: UCI "Seoul Bike Sharing Demand" 8,760시간(2017-12~2018-11). GitHub 저장소 `syedsharin/Seoul-Bike-Sharing-Demand-Prediction`에서 바로 읽습니다.
- 미니 프로젝트: 서울교통공사 승하차 인원과 기상청 서울(108) 관측 자료. 강의 자료의 CSV 두 개를 파일 창에 올려 씁니다.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. ★ 실습은 앞의 ★ 실습만 돌려도 됩니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**와, 미니 프로젝트에서 공휴일을 가려 주는 **holidays**를 설치합니다. 판다스, 사이파이, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib holidays

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch05/data/"
for f in [
    "asos_서울108_시간자료_2023_2026.csv",
    "서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 우연을 직접 만들어 보기

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 따릉이 데이터는 인터넷 주소에서 바로 읽습니다. 넘파이(NumPy), 판다스, 맷플롯립과 함께 과학 계산 라이브러리 사이파이(SciPy)의 `stats`를 씁니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다. ★ 실습은 앞의 ★ 실습만 돌려도 됩니다. 무작위를 쓰는 실습은 시드를 고정했지만, 라이브러리 버전에 따라 끝자리가 조금 달라질 수 있습니다.

#### [파이썬 실습 5-1 ★] 데이터 불러오기와 하루 단위로 묶기

따릉이 데이터를 읽어 운영하지 않은 날을 확인하고, 24시간이 모두 있는 날만 하루 단위 표로 묶으시오. 맑은 날과 비 온 날의 하루 대여량을 요약하고 그리시오.

① 데이터를 읽고 열 이름을 짧게 바꿉니다. ② 운영하지 않은 시간을 세고 빼냅니다. ③ 날짜별로 묶고 "비 온 날" 열을 만듭니다. ④ 두 무리를 점으로 그립니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                        # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

URL = ("https://raw.githubusercontent.com/syedsharin/"
       "Seoul-Bike-Sharing-Demand-Prediction/main/SeoulBikeData.csv")
df = pd.read_csv(URL, encoding="latin-1")
df.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",
              "dew", "solar", "rain", "snow", "season", "holiday", "functioning"]
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
print(df.shape)

off = df[df["functioning"] == "No"]                       # 운영하지 않은 시간
print("운영하지 않은 날:", off["date"].nunique(), "일,", len(off), "시간, 대여량 합", off["count"].sum())
h = df[df["functioning"] == "Yes"]
day = (h.groupby("date")
         .agg(count=("count", "sum"), rain=("rain", "sum"), temp=("temp", "mean"),
              season=("season", "first"), hours=("hour", "size"))
         .reset_index())
day = day[day["hours"] == 24]                             # 24시간이 다 있는 날만
day["rainy"] = day["rain"] >= 1.0                         # 하루 1mm 이상이면 비 온 날
day["weekend"] = day["date"].dt.dayofweek >= 5            # 토요일(5), 일요일(6)
print(len(day), "일")
print(day.groupby("rainy")["count"].agg(["size", "mean", "std"]).round(0))

whole = df.groupby("date").agg(count=("count", "sum"), season=("season", "first"))
print("가을 하루 평균  0을 그대로 두면:", round(whole.loc[whole["season"] == "Autumn", "count"].mean()),
      "/ 운영한 날만:", round(day.loc[day["season"] == "Autumn", "count"].mean()))

rng = np.random.default_rng(0)                            # 점이 겹치지 않게 옆으로 흩뜨리기
for k, (name, g) in enumerate(day.groupby("rainy")):
    plt.scatter(k + rng.uniform(-0.25, 0.25, len(g)), g["count"], s=10, color="#9a9a9a")
    plt.hlines(g["count"].mean(), k - 0.3, k + 0.3, color="#C8234B", linewidth=3)
plt.xticks([0, 1], ["맑은 날", "비 온 날"])
plt.ylabel("하루 대여량 (대)")
plt.show()

8,760행이 모두 들어왔습니다. 운영하지 않은 13일(295시간)의 대여량은 모두 0입니다. 이 0을 둔 채 가을 평균을 내면 19,670대, 빼면 22,175대입니다. 운영한 날 가운데 24시간이 다 있는 352일이 남았고, [표 5-1]의 숫자가 그대로 나옵니다. 코드가 그린 그래프는 [그림 @strip]과 같습니다. `encoding="latin-1"`은 파일 속 °C 기호 때문에 넣었습니다(3장).

#### [파이썬 실습 5-2 ☆] 동전을 10만 번 던져 p값 느끼기

공정한 동전을 열 번씩 10만 세트 던져, 앞면이 여섯 번 이상 나온 세트의 비율을 구하시오. 이항검정 p값도 구하시오.

In [ ]:
rng = np.random.default_rng(42)
heads = rng.binomial(n=10, p=0.5, size=100_000)          # 동전 열 번 던지기를 10만 세트
print("10번 중 6번 이상 앞면:", (heads >= 6).mean())
print("이항검정 p값(10번 중 6번):", round(stats.binomtest(6, 10, 0.5).pvalue, 3))
print("이항검정 p값(100번 중 60번):", round(stats.binomtest(60, 100, 0.5).pvalue, 3))
from math import comb
print("홍차 8잔을 우연히 모두 맞힐 확률: 1 /", comb(8, 4), "=", round(1 / comb(8, 4), 3))

counts = np.bincount(heads, minlength=11)
plt.bar(range(11), counts, color=["#C8234B" if k >= 6 or k <= 4 else "#9a9a9a" for k in range(11)])
plt.xlabel("10번 중 앞면 수")
plt.ylabel("세트 수")
plt.show()

앞면이 여섯 번 이상인 세트가 약 38%입니다. 뒷면 쪽으로 같은 만큼 치우친 경우(앞면 네 번 이하)도 있으니, 양쪽을 더하면 이항검정의 p값 0.754와 맞습니다. 그래프의 빨간 막대가 그 양쪽입니다. 1.2의 홍차 실험에서 8잔을 우연히 모두 맞힐 확률 1/70도 여기서 확인합니다.

#### [파이썬 실습 5-3 ★] 순열검정: 이름표를 섞어 우연의 세계 만들기

"비 옴/맑음" 이름표를 무작위로 다시 붙이는 일을 1만 번 되풀이해, 실제 차이만큼 벌어진 경우를 세시오.

① 실제 차이를 구합니다. ② 대여량 352개를 섞어 앞 82개를 "비 온 날"로 봅니다. ③ 1만 번 되풀이해 차이를 모읍니다. ④ 히스토그램을 그립니다.

In [ ]:
rainy = day.loc[day["rainy"], "count"]
dry = day.loc[~day["rainy"], "count"]
obs = dry.mean() - rainy.mean()                           # 실제 차이
rng = np.random.default_rng(42)
values = day["count"].to_numpy()
n_rainy = len(rainy)
diffs = np.empty(10_000)
for i in range(10_000):
    shuffled = rng.permutation(values)                    # 이름표를 섞는다
    diffs[i] = shuffled[n_rainy:].mean() - shuffled[:n_rainy].mean()
print(f"실제 차이 {obs:,.0f}대")
print(f"섞었을 때 차이의 표준편차 {diffs.std():,.0f}대")
print("실제만큼 벌어진 경우:", (np.abs(diffs) >= abs(obs)).sum(), "번 / 1만 번")

plt.hist(diffs, bins=50, color="#9a9a9a")
plt.axvline(obs, color="#C8234B", linewidth=3)
plt.xlabel("섞어서 만든 평균 차이 (대)")
plt.ylabel("횟수")
plt.show()

섞어 만든 차이의 표준편차는 1,267대이고, 실제만큼 벌어진 경우는 1만 번 가운데 3번입니다. p값은 약 0.0003입니다. 공식 없이 섞기만 해서 얻은 답입니다. 코드가 그린 히스토그램은 [그림 @perm_hist]과 같습니다.

> **[코드 읽기] `rng.permutation( )`**
>
> `np.random.default_rng(42)`는 시드 42로 난수 생성기를 만듭니다. 시드가 같으면 몇 번을 돌려도 같은 난수가 나옵니다. `rng.permutation(values)`는 배열의 순서를 무작위로 섞은 새 배열을 돌려줍니다. 원래 `values`는 바뀌지 않습니다.

#### [파이썬 실습 5-4 ★] t검정, 신뢰구간, 효과 크기

맑은 날과 비 온 날을 웰치 t검정으로 견주고, 차이의 95% 신뢰구간과 효과 크기 d를 구하시오. 하루 평균 대여량의 신뢰구간도 구하시오.

In [ ]:
res = stats.ttest_ind(dry, rainy, equal_var=False)       # 웰치 t검정
ci = res.confidence_interval(0.95)
print(f"t = {res.statistic:.2f}, 자유도 {res.df:.0f}, p = {res.pvalue:.5f}")
print(f"차이의 95% 신뢰구간 {ci.low:,.0f} ~ {ci.high:,.0f}대")

x = day["count"]
m, se = x.mean(), x.std() / np.sqrt(len(x))               # 표준오차 = s / √n
lo, hi = stats.t.interval(0.95, len(x) - 1, loc=m, scale=se)
print(f"하루 평균 {m:,.0f}대, 표준오차 {se:,.0f}대, 95% 신뢰구간 {lo:,.0f} ~ {hi:,.0f}대")

n1, n2 = len(dry), len(rainy)
sp = np.sqrt(((n1 - 1) * dry.var() + (n2 - 1) * rainy.var()) / (n1 + n2 - 2))  # 합동 표준편차
print(f"효과 크기 d = {obs / sp:.2f}, 맑은 날보다 {obs / dry.mean() * 100:.0f}% 적음")

plt.figure(figsize=(7, 2.2))
plt.errorbar([obs], [0], xerr=[[obs - ci.low], [ci.high - obs]], fmt="o", color="#C8234B",
             capsize=8, linewidth=3, markersize=10)
plt.axvline(0, color="#9a9a9a", linestyle="--")
plt.yticks([])
plt.xlabel("맑은 날 - 비 온 날, 하루 대여량 차이 (대)")
plt.xlim(-1000, 8000)
plt.show()

t = 3.89는 [예제 5-2]에서 손으로 구한 값과 같습니다. 자유도 148은 웰치 방식이 두 집단의 크기와 흔들림으로 계산한 값입니다. 차이의 신뢰구간 2,216~6,795대는 0을 품지 않습니다. 하루 평균의 표준오차 530대도 4.1의 손 계산과 같습니다. d = 0.46은 차이가 하루하루 흔들림의 절반쯤이라는 뜻입니다.

> **[코드 읽기] `stats.ttest_ind( )`**
>
> `stats.ttest_ind(dry, rainy, equal_var=False)`는 두 무리의 평균을 견주는 t검정입니다. `equal_var=False`가 "두 무리의 흔들림이 다를 수 있다"는 웰치 방식입니다. 결과에서 `.statistic`은 t, `.pvalue`는 p값, `.df`는 자유도이고, `.confidence_interval(0.95)`는 차이의 95% 신뢰구간을 돌려줍니다.

#### [파이썬 실습 5-5 ☆] 신뢰구간 100개와 부트스트랩

352일을 모집단으로 삼아 30일씩 뽑아 95% 신뢰구간을 100번 만들고, 진짜 평균을 품은 구간을 세시오. 부트스트랩으로 차이의 신뢰구간도 구하시오.

In [ ]:
true_mean = day["count"].mean()                           # 352일을 모집단으로 삼는다
rng = np.random.default_rng(7)
hit, lines = 0, []
for i in range(100):
    s = rng.choice(day["count"].to_numpy(), 30, replace=False)   # 30일 뽑기
    lo, hi = stats.t.interval(0.95, 29, loc=s.mean(), scale=s.std(ddof=1) / np.sqrt(30))
    hit += lo <= true_mean <= hi
    lines.append((lo, hi))
print("진짜 평균을 품은 신뢰구간:", hit, "개 / 100개")
boot = [rng.choice(dry, len(dry)).mean() - rng.choice(rainy, len(rainy)).mean()
        for _ in range(10_000)]                           # 부트스트랩
print("부트스트랩 95% 신뢰구간:", np.percentile(boot, [2.5, 97.5]).round(0))

for i, (lo, hi) in enumerate(lines):
    miss = not (lo <= true_mean <= hi)
    plt.vlines(i, lo, hi, color="#C8234B" if miss else "#9a9a9a", linewidth=3 if miss else 1.5)
plt.axhline(true_mean, color="black", linestyle="--")
plt.xlabel("신뢰구간 번호")
plt.ylabel("하루 평균 대여량 (대)")
plt.show()

100개 가운데 96개가 진짜 평균을 품었습니다. 95%에 가깝습니다([그림 @ci_net]). 부트스트랩(bootstrap)은 가진 표본을 작은 모집단으로 여기고, 그 안에서 되돌려 넣으며 다시 뽑기를 되풀이하는 방법입니다. 공식으로 구한 2,216~6,795대와 거의 같은 2,262~6,756대가 나왔습니다. 공식이 없는 중앙값이나 비율에도 같은 방법을 쓸 수 있습니다.

#### [파이썬 실습 5-6 ★] 달과 계절을 맞춰 다시 견주기

하루 평균기온이 0~10도인 날의 대여량을 달별로 견주고, 네 계절 각각에서 맑은 날과 비 온 날을 t검정으로 견주시오.

In [ ]:
band = day[(day["temp"] > 0) & (day["temp"] <= 10)]        # 하루 평균기온 0~10도인 날
print(band.groupby(band["date"].dt.to_period("M"))["count"].agg(["size", "mean"]).round(0))

rows = []
for s, name in [("Winter", "겨울"), ("Spring", "봄"), ("Summer", "여름"), ("Autumn", "가을")]:
    part = day[day["season"] == s]
    a = part.loc[~part["rainy"], "count"]
    b = part.loc[part["rainy"], "count"]
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    rows.append([name, len(a), round(a.mean()), len(b), round(b.mean()), round(a.mean() - b.mean()), round(p, 6)])
season = pd.DataFrame(rows, columns=["계절", "맑은 날", "평균", "비 온 날", "평균 ", "차이", "p값"])
print(season.to_string(index=False))

season.set_index("계절")[["평균", "평균 "]].plot.bar(color=["#9a9a9a", "#C8234B"], rot=0)
plt.legend(["맑은 날", "비 온 날"])
plt.ylabel("하루 평균 대여량 (대)")
plt.xlabel("")
plt.show()

같은 0~10도인데 11월(21일)의 하루 평균은 1월(10일)의 약 3배입니다. 계절 안에서 견주면 봄, 여름, 가을의 차이가 8,000대를 넘습니다. 겨울은 비 온 날이 10일뿐이라 p = 0.052입니다. [표 5-2]와 같고, 코드가 그린 막대그래프는 [그림 @season]와 같습니다.

> **[코드 읽기] `dt.to_period("M")`**
>
> 날짜 열에 `.dt.to_period("M")`을 붙이면 2018-11-03 같은 날짜가 "2018-11"이라는 달로 바뀝니다. 이것으로 `groupby( )`를 하면 달마다 묶입니다. "D"는 날, "W"는 주, "Y"는 해입니다.

#### [파이썬 실습 5-7 ★] 평일과 주말

평일과 주말의 하루 대여량을 웰치 t검정으로 견주고, 차이의 95% 신뢰구간을 그리시오.

In [ ]:
wd = day.loc[~day["weekend"], "count"]
we = day.loc[day["weekend"], "count"]
res = stats.ttest_ind(wd, we, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"평일 {len(wd)}일 {wd.mean():,.0f}대, 주말 {len(we)}일 {we.mean():,.0f}대")
print(f"차이 {wd.mean() - we.mean():,.0f}대, p = {res.pvalue:.3f}, 95% 신뢰구간 {ci.low:,.0f} ~ {ci.high:,.0f}대")

d = wd.mean() - we.mean()
plt.figure(figsize=(7, 2.2))
plt.errorbar([d], [0], xerr=[[d - ci.low], [ci.high - d]], fmt="o", color="#C8234B",
             capsize=8, linewidth=3, markersize=10)
plt.axvline(0, color="#9a9a9a", linestyle="--")
plt.yticks([])
plt.xlabel("평일 - 주말, 하루 대여량 차이 (대)")
plt.xlim(-1000, 8000)
plt.show()

[그림 @PLOT-5-4]과 같은 눈금으로 그렸습니다. 비 효과의 구간은 0에서 멀리 떨어져 있지만, 평일과 주말의 구간은 0을 걸칩니다. 그래서 "우연일 수 있다"고 씁니다. 동시에 구간의 오른쪽 끝(3,924대)이 보여 주듯, 꽤 큰 차이일 가능성도 남아 있습니다.

#### [파이썬 실습 5-8 ★] 다중검정 실험: 의미 없는 비교를 1,000번

352일을 동전 던지기로 두 편으로 나눠 t검정하는 일을 20번, 다시 1,000번 되풀이하고, p < 0.05가 몇 번 나오는지 세시오.

In [ ]:
rng = np.random.default_rng(2018)
ps = []
for _ in range(20):
    label = rng.random(len(day)) < 0.5                    # 동전으로 두 편 나누기
    ps.append(stats.ttest_ind(day["count"][label], day["count"][~label], equal_var=False).pvalue)
ps = np.array(ps)
print("20번 가운데 p < 0.05:", (ps < 0.05).sum(), "번")
count = 0
for _ in range(1000):
    label = rng.random(len(day)) < 0.5
    count += stats.ttest_ind(day["count"][label], day["count"][~label], equal_var=False).pvalue < 0.05
print("1,000번 가운데 p < 0.05:", count, "번")

plt.scatter(range(1, 21), ps, color=["#C8234B" if p < 0.05 else "#9a9a9a" for p in ps], s=60)
plt.axhline(0.05, color="#C8234B", linestyle="--")
plt.yscale("log")
plt.yticks([0.01, 0.05, 0.1, 0.5, 1], ["0.01", "0.05", "0.1", "0.5", "1"])
plt.xticks(range(1, 21))
plt.xlabel("의미 없이 나눈 검정 (번째)")
plt.ylabel("p값 (로그 눈금)")
plt.show()

20번 가운데 2번, 1,000번 가운데 49번(4.9%)이 "유의"했습니다. 아무 차이가 없어도 약 5%는 속는다는 뜻입니다. 코드가 그린 그래프는 [그림 @multi20]와 같습니다. 기준을 0.05 ÷ 20 = 0.0025로 바꾸어 다시 세어 보십시오.

#### [파이썬 실습 5-9 ☆] 큰 표본의 함정: 많이 모아도 치우치면

운영한 8,465시간을 모집단으로 삼아, 여름·가을 시간만 모은 큰 표본과 1년 전체에서 무작위로 뽑은 100시간 가운데 어느 쪽이 1년 평균에 가까운지 견주시오.

In [ ]:
pop = h["count"].to_numpy()                                    # 운영한 8,465시간 전체
big = h.loc[h["season"].isin(["Summer", "Autumn"]), "count"]   # 여름·가을만 모은 큰 표본
rng = np.random.default_rng(5)
small = rng.choice(pop, 100, replace=False)                    # 무작위 100시간
lo, hi = stats.t.interval(0.95, 99, loc=small.mean(), scale=small.std(ddof=1) / 10)
print(f"진짜 평균 {pop.mean():.1f}대")
print(f"큰 표본 {len(big):,}시간: {big.mean():.1f}대")
print(f"무작위 100시간: {small.mean():.1f}대 (95% 신뢰구간 {lo:.1f} ~ {hi:.1f})")
cover = 0
for _ in range(1000):
    s = rng.choice(pop, 100, replace=False)
    lo, hi = stats.t.interval(0.95, 99, loc=s.mean(), scale=s.std(ddof=1) / 10)
    cover += lo <= pop.mean() <= hi
print("무작위 100시간 신뢰구간 1,000개 가운데 진짜 평균을 품은 것:", cover, "개")

4,145시간을 모은 큰 표본은 진짜 평균 729.2대보다 253대(35%) 높게 짚었습니다. 무작위 100시간은 표본이 40분의 1인데도 38대 차이였고, 신뢰구간이 진짜 평균을 감쌌습니다. 무작위 100시간으로 신뢰구간을 1,000번 만들었더니 950개가 진짜 평균을 품었습니다. 1936년 여론조사를 따릉이로 다시 해 본 셈입니다.

#### [파이썬 실습 5-10 ☆] 카이제곱검정: 비 오는 시간과 바쁜 시간

시간 단위 데이터로 "비가 오는 시간인가"와 "바쁜 시간인가"의 교차표를 만들고 카이제곱검정을 하시오. 계절마다 바쁜 시간의 비율도 견주시오.

In [ ]:
hh = h.assign(busy=h["count"] > h["count"].median(), raining=h["rain"] > 0)
table = pd.crosstab(hh["raining"], hh["busy"])
print(table)
chi = stats.chi2_contingency(table)
print(f"카이제곱 = {chi.statistic:.1f}, p = {chi.pvalue:.1e}")
print(hh.groupby(["season", "raining"])["busy"].mean().unstack().round(3))

비와 무관하다면 비 오는 516시간 가운데 절반쯤인 258시간이 바빠야 하는데, 실제로는 38시간이었습니다. 계절마다 나눠도 비 오는 시간의 바쁜 비율은 0~15%로, 비가 오지 않는 시간(가을 72.1%, 여름 80.0%)보다 훨씬 낮습니다. 겨울은 바쁜 시간 자체가 4.2%뿐이라 견줄 것이 거의 없습니다.

#### [파이썬 실습 5-11 ☆] 상관계수와 기온 구간

기온과 대여량의 상관계수와 그 95% 신뢰구간을 구하고, 기온 구간별 평균을 견주시오.

In [ ]:
r = stats.pearsonr(day["temp"], day["count"])
print(f"기온-대여량 r = {r.statistic:.3f}, 95% 신뢰구간 {r.confidence_interval().low:.3f} ~ {r.confidence_interval().high:.3f}")
print(f"강수량-대여량 r = {stats.pearsonr(day['rain'], day['count']).statistic:.3f}")
bands = pd.cut(day["temp"], [-20, 0, 10, 20, 25, 28, 34])
print(day.groupby(bands, observed=True)["count"].agg(["size", "mean"]).round(0))

plt.scatter(day["temp"], day["count"], s=10, color="#9a9a9a")
hot = day["temp"] > 28
plt.scatter(day.loc[hot, "temp"], day.loc[hot, "count"], s=14, color="#C8234B")
plt.xlabel("하루 평균기온 (°C)")
plt.ylabel("하루 대여량 (대)")
plt.show()

r = 0.754로 강한 양의 상관입니다. 그런데 구간별 평균은 25~28도에서 26,368대로 가장 많고(4장은 4도 구간이라 24~28도, 26,630대), 28도를 넘는 36일은 22,579대로 줄어듭니다. 상관계수는 직선 관계만 잽니다. "더울수록 많이 탄다"는 한 문장은 28도에서 깨집니다. 그래서 상관계수는 늘 산점도와 함께 봅니다.

> **핵심 3줄**
>
> 순열검정은 이름표를 섞어 "우연이라면 이 정도 차이가 얼마나 자주 나오나"를 직접 세고, t검정은 같은 질문을 식으로 답합니다.
>
> p값만 보지 말고 신뢰구간과 효과 크기를 함께 보고, 여러 번 견줄수록 우연히 "유의"가 나온다는 점을 잊지 않습니다.
>
> 비 온 날의 차이는 섞기로 p ≈ 0.0003, t검정으로 t = 3.89, 신뢰구간 2,216~6,795대, d = 0.46이었고, 의미 없는 비교 1,000번 가운데 4.9%가 "유의"했습니다.

## 04절 계산 확인

책 04절의 손 계산을 파이썬으로 다시 해 봅니다.

### EQ-1 · (식 5-2) 표준오차

In [ ]:
s, n = 9951, 352
print("표준오차:", round(s / np.sqrt(n)))
print("표본을 4배로:", round(s / np.sqrt(4 * n)))

### EQ-2 · [예제 5-1] 여론조사 오차범위

In [ ]:
for n in [1000, 4000, 2_400_000]:
    print(n, "명: ±", round(1.96 * np.sqrt(0.5 * 0.5 / n) * 100, 2), "%p")

### EQ-3 · [예제 5-2] t통계량과 이름표를 섞은 세계

In [ ]:
m1, s1, n1 = 18539, 10025, 270      # 맑은 날
m2, s2, n2 = 14034, 8920, 82        # 비 온 날
se = np.sqrt(s1**2 / n1 + s2**2 / n2)
print("분모(표준오차):", round(se), " t =", round((m1 - m2) / se, 2))
print("이름표를 섞은 세계의 표준오차:", round(9951 * np.sqrt(1 / n1 + 1 / n2)))

### EQ-4 · (식 5-6) 효과 크기 d

In [ ]:
sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
print("합동 표준편차:", round(sp), " d =", round((m1 - m2) / sp, 2))

## [미니 프로젝트 5] 비 오는 평일에는 지하철 승객이 줄까


**질문.** 비가 오면 따릉이는 24% 덜 빌렸습니다. 그 사람들은 지하철을 탔을까요? 아니면 지하철 승객도 함께 줄었을까요? 비 오는 평일의 서울 지하철 승객 수를 보통 평일과 견주어, 그 차이가 우연인지, 얼마나 큰지 답하시오.

**데이터.** 한국 공공데이터 두 가지를 날짜로 잇습니다. 둘 다 공공누리 제1유형으로 출처를 밝히면 쓸 수 있습니다. 원자료가 크므로 수업에서는 미리 줄여 둔 강의 자료 파일 두 개를 씁니다. 13장의 종합 프로젝트도 같은 파일을 씁니다.

**[표 5-4] 미니 프로젝트 데이터**

| 파일 | 출처 | 크기 | 한 행 |
|---|---|---|---|
| 서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv | 서울교통공사 역별 일별 시간대별 승하차 인원(공공데이터포털) | 1,462행 × 22열 | 하루의 승차 또는 하차. 모든 역을 더한 시간대별 인원과 합계 |
| asos_서울108_시간자료_2023_2026.csv | 기상청 종관기상관측(ASOS) 서울 관측소(기상자료개방포털) | 33,241행 × 7열 | 한 시각의 기온, 강수량, 바람, 습도 |

> **[더 알아보기] ☆ 원자료를 직접 받아 합치기**
>
> 강의 자료 파일은 원자료를 줄인 것입니다. 직접 받아 보고 싶다면 공공데이터포털(data.go.kr)에서 "서울교통공사 역별 일별 시간대별 승하차인원 정보"를 검색해 해마다의 CSV 파일을 받습니다. 역 하나, 하루, 승차 또는 하차가 한 행이라 한 해에 20만 행(약 33MB)이 넘습니다. 날씨는 기상자료개방포털(data.kma.go.kr)에서 종관기상관측(ASOS)의 시간자료를 고르고, 지점은 서울(108)로 받습니다. 회원 가입이 필요하고, 기간을 나눠 여러 파일로 받게 됩니다.
>
> 받은 파일에는 함정이 셋 있습니다. 첫째, 한글이 cp949로 저장되어 있어 `encoding="cp949"`를 주어야 읽힙니다. 둘째, 해마다 열이 다릅니다. 2021년 파일은 늦은 시간이 "23시이후" 한 열이고, 2022년부터는 "23-24시간대"와 "24시이후" 두 열입니다. 셋째, 기간이 겹치는 파일이 함께 올라와 있기도 합니다. 노트북의 MP-0 셀이 이 세 가지를 처리해 강의 자료와 같은 모양의 파일을 만듭니다. 이 셀로 합친 2023~2024년 파일은 강의 자료 파일과 숫자가 모두 같았습니다. API(인증키로 받는 방법)도 있지만, 이 책의 실습에는 쓰지 않습니다.

### 데이터 파일 올리기

미니 프로젝트용으로 줄인 아래 두 파일이 강의 자료(책의 `ch05/data` 폴더)에 들어 있습니다. 코랩 왼쪽의 📁 파일 창에 끌어다 놓은 뒤 1단계부터 실행하십시오. 13장 종합 프로젝트 주제 B도 같은 파일을 씁니다.

- `서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv` (날짜·승하차마다 모든 역을 더한 파일)
- `asos_서울108_시간자료_2023_2026.csv` (기상청 서울 관측소 시간자료)

출처: 서울교통공사(공공데이터포털), 기상청 기상자료개방포털. 공공누리 제1유형에 따라 출처를 밝힙니다.

**☆ 원자료로 직접 해 보기.** 포털에서 받은 원자료 CSV를 파일 창에 올렸다면 아래 MP-0 셀이 강의 자료와 같은 모양의 파일 두 개(`지하철_직접_합친_파일.csv`, `날씨_직접_합친_파일.csv`)를 만듭니다. 그다음 1단계 코드 맨 위의 `SUB`, `WEATHER` 두 줄만 그 파일 이름으로 바꾸십시오. 원자료가 없으면 이 셀은 안내 문구만 찍고 끝납니다.

In [ ]:
# ☆ 포털에서 받은 원자료를 강의 자료와 같은 모양으로 합치기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
SUB_RAW = sorted(glob.glob("서울교통공사_역별*승하차인원*.csv"))   # 공공데이터포털에서 받은 파일
ASOS_RAW = sorted(glob.glob("OBS_ASOS_TIM_*.csv"))                 # 기상자료개방포털에서 받은 파일
if not SUB_RAW and not ASOS_RAW:
    print("포털에서 받은 파일이 없습니다. 강의 자료의 파일 두 개로 1단계를 하십시오.")
if SUB_RAW:
    parts = []
    for f in SUB_RAW:
        t = pd.read_csv(f, encoding="cp949")                        # 포털 파일의 한글은 cp949
        t = t.rename(columns={"수송일자": "날짜", "승하차구분": "구분"})
        t.columns = [c.replace("시간대", "시") for c in t.columns]   # 13-14시간대 → 13-14시
        late = [c for c in ["23-24시", "24시이후"] if c in t.columns]   # 늦은 시간 열은 해마다 다르다
        if late:
            t["23시이후"] = t[late].sum(axis=1)
            t = t.drop(columns=late)
        parts.append(t)
    raw = pd.concat(parts).drop_duplicates(["날짜", "역번호", "구분"], keep="last")  # 기간이 겹치는 파일
    hours = [c for c in raw.columns if c.endswith(("시", "이전", "이후"))]
    city = raw.groupby(["날짜", "구분"])[hours].sum()
    city["합계"] = city.sum(axis=1)
    city.reset_index().to_csv("지하철_직접_합친_파일.csv", index=False)
    print("지하철", len(SUB_RAW), "개 파일 →", city.shape, city.index[0][0], "~", city.index[-1][0])
if ASOS_RAW:
    wr = pd.concat([pd.read_csv(f, encoding="cp949") for f in ASOS_RAW])
    wr = wr.drop_duplicates("일시", keep="last").sort_values("일시")   # 기간이 겹치면 한 번만
    wr[["지점", "지점명", "일시", "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)"]].to_csv(
        "날씨_직접_합친_파일.csv", index=False)
    print("날씨", len(ASOS_RAW), "개 파일 →", len(wr), "시각", wr["일시"].iloc[0], "~", wr["일시"].iloc[-1])

**1단계: 데이터 신원 확인**([파이썬 실습 5-1] 응용). 두 파일을 읽어 기간과 빈칸을 확인하고, 하루 강수량을 구하시오. 달마다 하루 평균 승차 인원을 그려 이상한 값이 없는지 보시오.

In [ ]:
# 데이터 바꾸기: MP-0으로 직접 합친 파일을 쓸 때는 이 두 줄만 고칩니다.
SUB = "서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv"
WEATHER = "asos_서울108_시간자료_2023_2026.csv"

sub = pd.read_csv(SUB, parse_dates=["날짜"])
print(sub.shape, sub["날짜"].min().date(), "~", sub["날짜"].max().date(), "/ 빈칸", sub.isna().sum().sum())
on = sub[sub["구분"] == "승차"].set_index("날짜")             # 하루에 지하철을 탄 사람(승차)
w = pd.read_csv(WEATHER, parse_dates=["일시"])
w["강수량(mm)"] = w["강수량(mm)"].fillna(0)                    # 비가 안 온 시각은 빈칸으로 적혀 있다
rain = w.groupby(w["일시"].dt.normalize())["강수량(mm)"].sum()  # 하루 강수량
month = (on[["13-14시", "합계"]] / 10000).groupby(on.index.to_period("M")).mean().round(1)
print(month.loc["2024-02":"2024-06"])                         # 만 명, 하루 평균

month.index = month.index.strftime("%y.%m")
month["13-14시"].plot(marker="o", color="#C8234B", rot=0, figsize=(7, 3))
plt.ylabel("13~14시 승차 (만 명)")
plt.xlabel("")
plt.show()

2024년 4월 한 달만 13~14시 승차가 하루 평균 175.9만 명입니다. 다른 달은 20만 명 남짓입니다. 그래서 그달의 하루 합계도 635.2만 명으로 부풀었습니다. 1장의 미니 프로젝트에서 처음 만난 그 달입니다. 강의 자료를 만들며 생긴 값이 아니라 공개된 원자료에 그대로 있는 값입니다(4월 2일 신림역의 13~14시 승차 39,275명 등). "합계" 열도 이 칸을 그대로 더했으므로 믿을 수 없습니다. 이 프로젝트에서는 2024년 4월을 통째로 빼고, 그 결정을 기록해 둡니다. 3장의 따릉이 "운영하지 않은 날"과 같은 종류의 함정입니다. 3장의 미니 프로젝트는 정제를 연습하려고 같은 칸을 앞뒤 시간대의 평균으로 채우고 표시를 남겼습니다. 여기서는 비 온 날과 보통 날을 견주므로, 채워 넣은 숫자를 검정에 섞기보다 그달을 빼는 쪽을 골랐습니다. 날씨 파일의 강수량 빈칸은 "비가 오지 않은 시각"이라 0으로 채웠습니다.

**2단계: 기준선 만들기**([파이썬 실습 5-3], [5-4] 응용). 공휴일이 아닌 월~금요일만 남기고, 하루 강수량 10mm 이상인 날을 "비 온 평일"로 정하시오. 비 온 평일과 그 밖의 평일의 평균을 그대로 견주시오.

비의 기준은 데이터를 보기 전에 정했습니다. 기후 연구에서 "비가 꽤 온 날"을 셀 때 흔히 쓰는 일강수량 10mm를 빌렸습니다. 공휴일은 `holidays` 라이브러리로 설·추석 연휴와 대체 공휴일까지 뺐습니다.

In [ ]:
import holidays
kr = holidays.KR(years=[2023, 2024])                          # 설·추석 연휴, 대체 공휴일까지
d = (on["합계"] / 10000).to_frame("승차").join(rain.rename("강수량"), how="inner")
d = d[d.index.to_period("M") != pd.Period("2024-04")]         # 13~14시 칸이 이상한 달은 뺀다
d["평일"] = [t.weekday() < 5 and t not in kr for t in d.index]
wk = d[d["평일"]].copy()                                       # 공휴일이 아닌 월~금
wk["비"] = wk["강수량"] >= 10                                   # 미리 정한 기준: 하루 10mm 이상
a, b = wk.loc[wk["비"], "승차"], wk.loc[~wk["비"], "승차"]
print("평일", len(wk), "일 가운데 비 온 날", len(a), "일")
print(f"비 온 평일 {a.mean():.1f}만 명, 그 밖의 평일 {b.mean():.1f}만 명, 차이 {a.mean() - b.mean():.1f}만 명")
print("p =", f"{stats.ttest_ind(a, b, equal_var=False).pvalue:.6f}")
g = wk.groupby(wk.index.month).agg(비온날=("비", "sum"), 승차=("승차", "mean"))
print(g.round(1).T.to_string())
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(g.index, g["승차"], color=["#C8234B" if m in (7, 8) else "#9a9a9a" for m in g.index])
ax.set_ylim(440, 520)
ax.set_xticks(g.index)
ax.set_xlabel("월")
ax.set_ylabel("평일 하루 승차 (만 명)")
plt.show()

평일 472일 가운데 비 온 날은 47일입니다. 그대로 견주면 비 온 평일이 18.0만 명 적고, p값도 아주 작습니다. 그런데 표를 보면 비 온 날 47일 가운데 22일이 7월과 8월에 몰려 있습니다. 그리고 8월(460.9만 명)과 7월(479.2만 명)은 승객이 적은 달입니다. 이 달 평균에도 비 온 날이 섞여 있지만, 방학과 휴가가 겹치니 비가 없어도 적은 달일 가능성이 큽니다. 3단계에서 같은 달끼리 견주는 까닭입니다. 따릉이에서는 계절이 비의 효과를 가렸습니다. 여기서는 반대로, 달이 비의 효과를 부풀리고 있을 수 있습니다.

**3단계: 이 장의 기법 쓰기**([파이썬 실습 5-3], [5-4] 응용). 평일마다 "같은 달, 같은 요일의 보통 날"(중앙값)을 구하고, 그 보통 날과 견준 차이(%)를 만드시오. 비 온 평일과 그 밖의 평일의 차이를 웰치 t검정, 순열검정, 신뢰구간, 효과 크기로 나타내시오.

`groupby( )` 뒤의 `transform("median")`은 묶음마다 중앙값을 구해, 원래 행마다 자기 묶음의 값을 붙여 줍니다. 그래서 7월의 월요일에는 "7월 월요일의 보통 날" 값이 붙습니다. 달과 요일이 만든 차이를 이렇게 빼고 견줍니다.

In [ ]:
wk["보통"] = wk.groupby([wk.index.month, wk.index.dayofweek])["승차"].transform("median")
wk["차이(%)"] = (wk["승차"] / wk["보통"] - 1) * 100            # 같은 달·같은 요일의 보통 날과 견준 차이
x, y = wk.loc[wk["비"], "차이(%)"], wk.loc[~wk["비"], "차이(%)"]
res = stats.ttest_ind(x, y, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"비 온 평일 {x.mean():.2f}%, 그 밖의 평일 {y.mean():.2f}%")
print(f"웰치 t검정 p = {res.pvalue:.4f}, 차이의 95% 신뢰구간 {ci.low:.2f} ~ {ci.high:.2f}%p")
rng = np.random.default_rng(42)
v, lab = wk["차이(%)"].to_numpy(), wk["비"].to_numpy()
diffs = np.empty(10_000)
for i in range(10_000):
    s = rng.permutation(lab)                                  # 비 이름표를 섞는다
    diffs[i] = v[s].mean() - v[~s].mean()
print("순열검정 p =", (np.abs(diffs) >= abs(x.mean() - y.mean())).mean())
sp = np.sqrt(((len(x) - 1) * x.var() + (len(y) - 1) * y.var()) / (len(x) + len(y) - 2))
print(f"효과 크기 d = {(y.mean() - x.mean()) / sp:.2f}")

plt.hist(y, bins=30, color="#9a9a9a", alpha=0.8, density=True, label="그 밖의 평일")
plt.hist(x, bins=15, color="#C8234B", alpha=0.6, density=True, label="비 온 평일")
plt.axvline(0, color="black", linestyle="--")
plt.xlabel("같은 달·같은 요일의 보통 날과 견준 차이 (%)")
plt.ylabel("밀도")
plt.legend()
plt.show()

**4단계: 결과 해석**. 차이를 사람 수로 바꾸고, 다른 기준(1mm)으로 한 번 더 확인하시오. 검사를 두 번 했다는 것도 함께 쓰시오.

In [ ]:
usual = wk["보통"].median()
gap = (x.mean() - y.mean()) / 100 * usual
print(f"보통 평일 승차 {usual:.1f}만 명, 비 오는 평일에 줄어드는 사람 약 {abs(gap):.1f}만 명")
wk["비1"] = wk["강수량"] >= 1                                  # 다른 기준(1mm)으로 한 번 더: 검사 횟수 2번
x1, y1 = wk.loc[wk["비1"], "차이(%)"], wk.loc[~wk["비1"], "차이(%)"]
print(f"1mm 기준: 비 온 평일 {len(x1)}일 {x1.mean():.2f}%, 그 밖 {y1.mean():.2f}%, "
      f"p = {stats.ttest_ind(x1, y1, equal_var=False).pvalue:.3f}")

달과 요일을 맞추자 차이는 18만 명에서 약 10만 명으로 줄었습니다. 비 온 평일은 보통 날보다 평균 2.18% 적었고, 그 밖의 평일은 0.17% 적었습니다. 두 무리의 차이는 2%p쯤입니다(95% 신뢰구간 -3.24~-0.78%p, p = 0.0018, 순열검정 p = 0.0004). 우연으로 보기는 어렵습니다. 효과 크기 d는 0.61로 "중간"을 넘지만, 이것은 평일끼리의 흔들림이 아주 작아서입니다. 원래 단위로는 보통 평일 약 498만 명 가운데 10만 명, 2%입니다. 따릉이의 24%와 견주면 아주 작은 변화입니다. 1mm 기준으로 다시 보면 차이는 더 작아지고 p = 0.044입니다. 두 번 검사했으니 본페로니 기준(0.025)으로는 이 결과를 "유의"하다고 말하기 어렵습니다.

**5단계: 한계 쓰기**. 이 분석이 아직 모르는 것을 [표 5-3]처럼 세 가지 이상 쓰시오. 예를 들어 하루 합계는 비가 출근 시간에 왔는지 밤에 왔는지 구별하지 못합니다. 강의 자료 파일에는 서울교통공사의 1~8호선만 들어 있으므로 9호선과 버스는 빠져 있습니다. 2023~2024년 두 해뿐이고, 비 온 평일 47일 가운데 절반이 여름에 있습니다.

**제출물.** 그림 2장(달별 승차 인원과 이상한 달, 보통 날과 견준 차이의 분포), 숫자 표 1개(그대로 견준 차이와 달·요일을 맞춘 차이, 각각의 p값과 신뢰구간), 결론 5문장 이내.

**[표 5-5] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 2024년 4월의 이상한 값과 "합계" 열까지 확인하고, 뺀 까닭을 기록함 | 이상한 달을 뺐으나 까닭을 쓰지 않음 | 점검 없이 합계를 그대로 씀 |
| 기준선 비교(20%) | 그대로 견준 차이와 달·요일을 맞춘 차이를 둘 다 보이고 왜 다른지 설명함 | 둘 가운데 하나만 보임 | 비교 없이 평균만 씀 |
| 기법 적용(30%) | 검정, 신뢰구간, 효과 크기를 모두 맞게 구하고, 기준을 미리 정함 | 검정은 맞으나 신뢰구간이나 효과 크기가 없음 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | "유의하다"와 "크다"를 구별하고, 검사 횟수와 한계를 셋 이상 씀 | 한계를 한두 가지만 씀 | 숫자만 나열함 |

**교수용 정답 예시.** 핵심 숫자는 위 3~4단계와 같습니다. 그대로 견주면 18.0만 명 차이(p = 0.00001)이고, 달·요일을 맞추면 2.18% 대 0.17%(p = 0.0018, d = 0.61), 곧 하루 약 10만 명입니다. 전체 풀이는 교수용 노트북에 있습니다.

## ✏️ [연습문제 5-6] 눈 온 겨울날 견주기

따릉이 데이터에서 24시간이 모두 있는 겨울날 가운데 눈이 온 날과 오지 않은 날의 대여량을 웰치 t검정과 순열검정으로 견주고, 두 p값을 나란히 적으시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
winter = day[day["season"] == "Winter"].copy()
winter["snowy"] = winter["date"].isin(h.loc[h["snow"] > 0, "date"])
a = winter.loc[winter["snowy"], "count"]
b = winter.loc[~winter["snowy"], "count"]
res = stats.ttest_ind(b, a, equal_var=False)
print(f"눈 온 날 {len(a)}일 {a.mean():,.0f}대, 안 온 날 {len(b)}일 {b.mean():,.0f}대")
print(f"웰치 t = {res.statistic:.2f}, p = {res.pvalue:.4f}")
rng = np.random.default_rng(42)
v, n_a = winter["count"].to_numpy(), len(a)
diffs = []
for _ in range(10_000):
    s = rng.permutation(v)
    diffs.append(s[n_a:].mean() - s[:n_a].mean())
print("순열검정 p =", (np.abs(diffs) >= abs(b.mean() - a.mean())).mean())